# Interrupts Tutorial

Previously, you checked the button over and over inside loop(). This is called **polling**. It works, but your code can only notice a button press when it gets around to checking. If loop() is busy (for example, sitting in a delay()), presses can be missed.

**Interrupts** solve this problem. An interrupt pauses the code that is currently running to execute a short function, then returns to where it left off in loop(). The main code will not resume until the interrupt function is finished. The microcontroller's hardware watches the pin for you, so the interrupt activates the moment its condition is met, no matter what loop() is doing. Timer overflows are another type of interrupt, which you will see in Exercise 4.

Think of it like the difference between getting up to check the front door every few minutes (polling) and installing a doorbell (interrupt).

Not every pin can trigger an interrupt. On an Arduino Uno or Nano, only pins 2 and 3 can be used.

**IMPORTANT:** If you implement an interrupt which modifies a variable (perhaps one you're using to track the state of a game of Pong), ensure that variable is declared using **volatile**. Section 2 explains why.

## 1 Your First Interrupt

Suppose we want to trigger an interrupt upon pressing a button. As soon as the pin the button is attached to changes, we want to run a function called buttonInterrupt().

This code can be pasted into the starter.ino file to demonstrate how button interrupts work. After you understand how they work, you can use this method to add interrupt functionality to the skeleton.

### Includes
Add the following lines of code before the setup section in the Arduino IDE: 


In [ ]:
#include <Wire.h>
#include <Adafruit_GFX.h>
#include <Adafruit_SSD1306.h>

### Definitions
Add the following lines of code before the setup section in the Arduino IDE: 


In [ ]:
// This code will turn the built-in LED off as soon as the button is pressed
const int BTN_PAUSE = 2;   // Must be an interrupt-capable pin (pins 2 and 3 on an Uno/Nano)

void buttonInterrupt() {
    // This is the code that will be executed after triggering the interrupt
    digitalWrite(LED_BUILTIN, LOW);
}

### Setup Code

In [ ]:
pinMode(LED_BUILTIN, OUTPUT);
pinMode(BTN_PAUSE, INPUT_PULLUP);

/* INPUT_PULLUP holds the pin HIGH when the button is not pressed.
Pressing the button connects the pin to ground, pulling it LOW.
This is why our buttons are "active LOW". */

attachInterrupt(digitalPinToInterrupt(BTN_PAUSE), buttonInterrupt, FALLING);

/* attachInterrupt takes 3 values:
1. digitalPinToInterrupt(pin) declares which pin to attach the interrupt to.
2. The interrupt function, which must take no inputs and return void.
3. The interrupt configuration:
   CHANGE will trigger the interrupt when the pin changes state.
   FALLING will trigger when the pin changes from 1 to 0. We want to use this
   for this exercise as our buttons are active LOW, so FALLING means "just pressed".
   RISING will trigger when the pin changes from 0 to 1, meaning "just released". */

### Loop Code

In [ ]:
digitalWrite(LED_BUILTIN, HIGH);
delay(1000);

### Try It

Press the button a few times and watch the LED, then answer the following:

1. The LED does not stay off for the same amount of time on every press. Why not?
2. If you had used polling from Exercise 2 to check the button inside loop(), what would happen to a press made during the delay(1000)?
3. After you've confirmed the interrupt works, remove the delay(1000) statement from loop(). What happens now when you press the button? Did the interrupt stop working?

An interrupt function is often called an **ISR** (Interrupt Service Routine). When creating buttonInterrupt() or any ISR, keep it short and quick so the code can return to the main loop as quickly as possible. Do not use loops or delays. The delay() function relies on interrupts itself, so it will not work inside an ISR. For the same reason, do not draw to the display or print to Serial inside an ISR, since the OLED screen communicates using the Wire library, which also relies on interrupts.

## 2 Volatile Variables and the Flag Pattern

Since interrupt functions must be short, they usually shouldn't do the real work themselves. Instead, the ISR sets a variable (often called a **flag**) to record that something happened. Then loop() checks that variable and does the slower work, like updating the display. This is how the pause button in Pong will work.

Any variable changed inside an ISR must be declared **volatile**. Normally, the compiler assumes a variable can only change where your code changes it, so it may reuse a saved copy instead of re-reading it. Since an interrupt can change the variable at any moment, volatile tells the compiler to re-read the real value every time it is used.

Replace the Definitions, Setup and Loop code from Section 1 with the code below. This version counts button presses and prints the count to the Serial Monitor.

### Definitions

In [ ]:
const int BTN_PAUSE = 2;
volatile int pressCount = 0;   // Changed inside the ISR, so it must be volatile

void buttonInterrupt() {
    pressCount++;   // Short and quick: just record that a press happened
}

### Setup Code

In [ ]:
Serial.begin(9600);
pinMode(BTN_PAUSE, INPUT_PULLUP);
attachInterrupt(digitalPinToInterrupt(BTN_PAUSE), buttonInterrupt, FALLING);

### Loop Code

In [ ]:
static int lastCount = 0;   // static keeps its value between runs of loop()

/* An int is 2 bytes on the Uno, but the processor reads it one byte at a time.
If the interrupt fires halfway through reading pressCount, we could get a garbled value.
Briefly turning interrupts off while we copy it prevents this. */
noInterrupts();
int count = pressCount;
interrupts();

if (count != lastCount) {
    Serial.print("Presses: ");
    Serial.println(count);
    lastCount = count;
}

Open the Serial Monitor (9600 baud) and slowly press the button 10 times. Does the final count match the number of times you pressed it?

You will likely find that the count sometimes jumps by 2, 3, or more on a single press. This is called **button bounce**. When the metal contacts inside a button close, they physically bounce against each other for a few milliseconds, so the pin flickers between HIGH and LOW several times. Interrupts are fast enough to catch every one of these flickers, and each one counts as a FALLING edge.

In Pong, this means a single press of the pause button could pause and unpause the game several times. The next section fixes this.

## 3 Debouncing

To fix button bounce, the ISR should ignore any trigger that happens too soon after the last real press. We can use millis() to check how much time has passed. Reading millis() inside an ISR is fine; it just won't count up while the ISR is running.

Replace the Definitions code from Section 2 with the code below and fill in the missing lines. Keep the Setup and Loop code from Section 2.

In [ ]:
const int BTN_PAUSE = 2;
const unsigned long DEBOUNCE_MS = 200;   // Triggers closer together than this are treated as bounce

volatile int pressCount = 0;
volatile unsigned long lastPressTime = 0;

void buttonInterrupt() {
    unsigned long now = millis();

    if (now - lastPressTime > DEBOUNCE_MS) {
        // This is a real press, not a bounce. What two things should happen here?

    }
    // If the if statement was skipped, the trigger was a bounce and is ignored
}

Test your code by pressing the button 10 times again. The count should now go up by exactly one on each press.

Then try changing DEBOUNCE_MS to 5, and then to 1000. What happens in each case, and why?

## 4 Pause Button

In the next section we will be building pong! It's important for any game to have a pause button. The first press of the pause button should freeze the game, and the second press should resume it.

Using the flag pattern, the ISR only flips a volatile bool called paused. Then loop() checks paused. If the game is paused, loop() draws a pause screen and skips the rest of the game code, so the ball and paddles stop moving.

This code can be added to the corresponding sections of the skeleton to add pause functionality. Remember to also add the pinMode and attachInterrupt lines from Section 1 to setup(), changing buttonInterrupt to pauseInterrupt.

### Definitions

In [ ]:
const int BTN_PAUSE = 2;   // Skip this line if the skeleton already defines BTN_PAUSE
const unsigned long DEBOUNCE_MS = 200;

volatile bool paused = false;
volatile unsigned long lastPauseTime = 0;

void pauseInterrupt() {
    /* Use your debounce check from Section 3.
    Instead of counting presses, flip the value of paused. */

}

### Loop Code

In [ ]:
if (paused) {
    // Draw a "PAUSED" message in the Serial Monitor. Remember, drawing belongs in loop(), not the ISR!

    return;   // Skip the rest of loop() so the ball and paddles freeze
}

// The rest of your game code, in the next section, (paddle movement, ball physics, scoring) goes below this

The return statement ends the current run of loop() early. Since loop() is called again right away, the game keeps checking whether it is still paused, and picks up where it left off as soon as paused becomes false.

## 5 Pause Design

Your pause button works, but there are still a few details to decide on.

Currently, the pause button can be pressed at any time. Should it work while the game is waiting for players, or on the game over screen? Change your game so that pausing only has an effect while the game is being played.

Jumping straight back into a fast-moving game can catch players off guard. Add a 3-2-1 countdown on the display when the game resumes. This countdown belongs in loop(), not in the ISR. Why? (Hint: you'll need a regular, non-volatile variable to remember whether the game was paused on the previous run of loop().)

If you finish early, think about what else in Pong could be controlled by an interrupt. On an Uno, pin 3 can also be used for interrupts.

---

<div style="display:flex; justify-content:space-between;">
  <span><b>Back: </b> <a href="02_analog_and_uart_tutorial.ipynb">Activity 2: Analog and UART</a></span>
  <span><b><a href="../../README.md">Top</a></b></span>
  <span><b>Next: </b> <a href="../README.md">README</a></span>
</div>